# **[Cycle GAN Pipeline]**

# **[Setup]**

In [ ]:
import os
import math
import random
import glob
import zipfile
from datetime import timedelta

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 50
BUFFER_SIZE = 1024
SEED = 42
PAINTER = 'vangogh' # options: ['monet', 'vangogh']

In [ ]:
drive.mount('/content/drive')

ZIP_PATH = f'/content/drive/MyDrive/CI/{PAINTER}.zip'

BASE_INPUT = f'/content/{PAINTER}'

if PAINTER == 'monet':
  PAINTING_TFREC = os.path.join(BASE_INPUT, 'monet_tfrec')
  PHOTO_TFREC = os.path.join(BASE_INPUT, 'photo_tfrec')
  PAINTING_JPG = os.path.join(BASE_INPUT, 'monet_jpg')
  PHOTO_JPG = os.path.join(BASE_INPUT, 'photo_jpg')

elif PAINTER == 'vangogh':
  BASE_INPUT = f'/content/{PAINTER}/vangogh2photo/'
  PAINTING_TFREC = None
  PHOTO_TFREC = None
  PAINTING_JPG = os.path.join(BASE_INPUT, 'trainA')
  PHOTO_JPG = os.path.join(BASE_INPUT, 'trainB')

OUTPUT_DIR = f'results/{PAINTER}'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("SEED SET SUCCESSFULLY!")

In [ ]:
# Try to connect to TPU if available
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
except Exception:
    # Fallback to GPU or CPU
    strategy = tf.distribute.get_strategy()

# Print device information
print('Number of replicas in sync:', strategy.num_replicas_in_sync)

# **[DATA LOAD]**

In [ ]:
if not os.path.exists(BASE_INPUT):
    print("Extracting data from Drive... this may take a minute.")
    os.makedirs(BASE_INPUT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
      zip_ref.extractall(BASE_INPUT)
    print("Extraction complete!")
else:
    print("Data already extracted.")

In [ ]:
def _tfrecord_example_description():
    feature_description = {
        'image': tf.io.FixedLenFeature([], tf.string)
    }
    return feature_description

# Parse TFRecord example
def parse_tfrecord(example_proto):
    features = tf.io.parse_single_example(example_proto, _tfrecord_example_description())
    image = tf.image.decode_jpeg(features['image'], channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method='bicubic')
    image = (image * 2.0) - 1.0
    return image

# Load TFRecord dataset
def load_tfrecord_dataset(tfrecord_dir):
    pattern = os.path.join(tfrecord_dir, '*.tfrec')
    files = tf.io.gfile.glob(pattern)
    dataset = tf.data.TFRecordDataset(files, num_parallel_reads=AUTOTUNE)
    dataset = dataset.map(parse_tfrecord, num_parallel_calls=AUTOTUNE)
    return dataset

# Load JPG dataset
def load_jpg_dataset(jpg_dir):
    pattern = os.path.join(jpg_dir, '*.jpg')
    files = tf.io.gfile.glob(pattern)
    files.sort()
    file_path_tensor = tf.constant(files, dtype=tf.string)
    path_ds = tf.data.Dataset.from_tensor_slices(file_path_tensor)
    def _load(path):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method='bicubic')
        image = (image * 2.0) - 1.0
        return image

    dataset = path_ds.map(_load, num_parallel_calls=AUTOTUNE)
    return dataset

In [ ]:
# Prepare datasets
painting_ds = load_jpg_dataset(PAINTING_JPG)
photo_ds = load_jpg_dataset(PHOTO_JPG)

# Shuffle and batch
painting_ds_train = painting_ds.shuffle(BUFFER_SIZE, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
photo_ds_train = photo_ds.shuffle(BUFFER_SIZE, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = tf.data.Dataset.zip((photo_ds_train, painting_ds_train))
train_ds = train_ds.prefetch(AUTOTUNE)

print('Painting samples (approx):', sum(1 for _ in painting_ds.take(5)) * (len(list(tf.io.gfile.glob(os.path.join(PAINTING_JPG, '*.jpg')))) // 5 if tf.io.gfile.exists(PAINTING_JPG) else BATCH_SIZE))
print('Photo samples (approx):', sum(1 for _ in photo_ds.take(5)) * (len(list(tf.io.gfile.glob(os.path.join(PHOTO_JPG, '*.jpg')))) // 5 if tf.io.gfile.exists(PHOTO_JPG) else BATCH_SIZE))


# **[EDA]**
We visualize a few paintings and photos to check distributions and dynamic ranges.

In [ ]:
def show_batch(dataset, title):
    images = next(iter(dataset.unbatch().batch(9)))
    plt.figure(figsize=(8,8))
    for i in range(min(9, images.shape[0])):
        plt.subplot(3,3,i+1)
        img = (images[i].numpy() + 1.0) / 2.0
        plt.imshow(np.clip(img, 0, 1))
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

# Show painting samples
show_batch(painting_ds.batch(9), 'Painting samples')

# Show photo samples
show_batch(photo_ds.batch(9), 'Photo samples')

# **[CycleGAN Architecture]**
We define the generator as a ResNet‑style encoder‑residual‑decoder and the discriminator as a PatchGAN.

In [ ]:
# Define reflection padding layer
class ReflectionPadding2D(layers.Layer):
    def __init__(self, padding=(1, 1)):
        super().__init__()
        self.padding = padding
    def call(self, inputs):
        pad_top, pad_left = self.padding
        pad_bottom, pad_right = self.padding
        return tf.pad(inputs, [[0,0],[pad_top,pad_bottom],[pad_left,pad_right],[0,0]], mode='REFLECT')

# Define residual block
def residual_block(x, filters):
    y = ReflectionPadding2D((1,1))(x)
    y = layers.Conv2D(filters, 3, strides=1, padding='valid')(y)
    y = layers.InstanceNormalization()(y)
    y = layers.Activation('relu')(y)
    y = ReflectionPadding2D((1,1))(y)
    y = layers.Conv2D(filters, 3, strides=1, padding='valid')(y)
    y = layers.InstanceNormalization()(y)
    y = layers.add([x, y])
    return y

# Define instance normalization for TF without addons
class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon
    def build(self, input_shape):
        self.scale = self.add_weight(name='scale', shape=input_shape[-1:], initializer='ones', trainable=True)
        self.offset = self.add_weight(name='offset', shape=input_shape[-1:], initializer='zeros', trainable=True)
    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1,2], keepdims=True)
        normalized = (x - mean) / tf.sqrt(var + self.epsilon)
        return self.scale * normalized + self.offset

layers.InstanceNormalization = InstanceNormalization

# Define generator
def build_generator(image_size=IMAGE_SIZE, residual_blocks=6, base_filters=64):
    inputs = layers.Input(shape=(image_size, image_size, 3))
    x = ReflectionPadding2D((3,3))(inputs)
    x = layers.Conv2D(base_filters, 7, strides=1, padding='valid')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(base_filters*2, 3, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(base_filters*4, 3, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.Activation('relu')(x)
    for _ in range(residual_blocks):
        x = residual_block(x, base_filters*4)
    x = layers.Conv2DTranspose(base_filters*2, 3, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2DTranspose(base_filters, 3, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.Activation('relu')(x)
    x = ReflectionPadding2D((3,3))(x)
    x = layers.Conv2D(3, 7, strides=1, padding='valid', activation='tanh')(x)
    return keras.Model(inputs, x, name='generator')

# Define PatchGAN discriminator
def build_discriminator(image_size=IMAGE_SIZE, base_filters=64):
    inputs = layers.Input(shape=(image_size, image_size, 3))
    x = layers.Conv2D(base_filters, 4, strides=2, padding='same')(inputs)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2D(base_filters*2, 4, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2D(base_filters*4, 4, strides=2, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2D(base_filters*8, 4, strides=1, padding='same')(x)
    x = layers.InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2D(1, 4, strides=1, padding='same')(x)
    return keras.Model(inputs, x, name='discriminator')

### **[[Losses and Optimizers]]**

In [ ]:
# Define losses
adv_loss_fn = keras.losses.MeanSquaredError()

# Define cycle consistency loss scale
LAMBDA_CYCLE = 10.0

# Define identity loss scale
LAMBDA_ID = 0.5

# Build models under distribution strategy
with strategy.scope():
    G = build_generator()
    F = build_generator()
    D_X = build_discriminator()
    D_Y = build_discriminator()
    g_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
    f_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
    dx_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
    dy_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)

# Define generator loss
def generator_loss(fake_logits):
    valid = tf.ones_like(fake_logits)
    loss = adv_loss_fn(valid, fake_logits)
    return loss

# Define discriminator loss
def discriminator_loss(real_logits, fake_logits):
    valid = tf.ones_like(real_logits)
    fake = tf.zeros_like(fake_logits)
    real_loss = adv_loss_fn(valid, real_logits)
    fake_loss = adv_loss_fn(fake, fake_logits)
    total = (real_loss + fake_loss) * 0.5
    return total

### **[[Training Step]]**

In [ ]:
# Define single training step
@tf.function
def train_step(real_x, real_y):
    with tf.GradientTape(persistent=True) as tape:
        fake_y = G(real_x, training=True)
        cycled_x = F(fake_y, training=True)
        fake_x = F(real_y, training=True)
        cycled_y = G(fake_x, training=True)
        same_x = F(real_x, training=True)
        same_y = G(real_y, training=True)
        disc_real_x = D_X(real_x, training=True)
        disc_real_y = D_Y(real_y, training=True)
        disc_fake_x = D_X(fake_x, training=True)
        disc_fake_y = D_Y(fake_y, training=True)
        g_loss_adv = generator_loss(disc_fake_y)
        f_loss_adv = generator_loss(disc_fake_x)
        cycle_loss = tf.reduce_mean(tf.abs(real_x - cycled_x)) + tf.reduce_mean(tf.abs(real_y - cycled_y))
        cycle_loss = LAMBDA_CYCLE * cycle_loss
        id_loss = tf.reduce_mean(tf.abs(real_x - same_x)) + tf.reduce_mean(tf.abs(real_y - same_y))
        id_loss = LAMBDA_ID * LAMBDA_CYCLE * id_loss
        g_total = g_loss_adv + cycle_loss + id_loss
        f_total = f_loss_adv + cycle_loss + id_loss
        dx_loss = discriminator_loss(disc_real_x, disc_fake_x)
        dy_loss = discriminator_loss(disc_real_y, disc_fake_y)
    g_grads = tape.gradient(g_total, G.trainable_variables)
    f_grads = tape.gradient(f_total, F.trainable_variables)
    dx_grads = tape.gradient(dx_loss, D_X.trainable_variables)
    dy_grads = tape.gradient(dy_loss, D_Y.trainable_variables)
    g_optimizer.apply_gradients(zip(g_grads, G.trainable_variables))
    f_optimizer.apply_gradients(zip(f_grads, F.trainable_variables))
    dx_optimizer.apply_gradients(zip(dx_grads, D_X.trainable_variables))
    dy_optimizer.apply_gradients(zip(dy_grads, D_Y.trainable_variables))
    return {
        'g_total': g_total,
        'f_total': f_total,
        'dx_loss': dx_loss,
        'dy_loss': dy_loss,
        'cycle_loss': cycle_loss,
        'id_loss': id_loss
    }

# **[Training Loop]**

In [ ]:
from tqdm.notebook import tqdm

def train(dataset, epochs=EPOCHS):
    steps_per_epoch = dataset.cardinality().numpy()
    if steps_per_epoch < 0:
        steps_per_epoch = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        pbar = tqdm(enumerate(dataset), total=steps_per_epoch, desc="Training")

        for step, (x_batch, y_batch) in pbar:
            losses = train_step(x_batch, y_batch)

            pbar.set_postfix({
                'g_tot': f"{losses['g_total']:.2f}",
                'f_tot': f"{losses['f_total']:.2f}",
                'dx': f"{losses['dx_loss']:.2f}",
                'dy': f"{losses['dy_loss']:.2f}",
                'cycle': f"{losses['cycle_loss']:.2f}",
                'id': f"{losses['id_loss']:.2f}"
            })

# Run training
train(train_ds, epochs=EPOCHS)

# **[Inference and Image Export]**
We transform the provided photos into painter style using the trained generator `G` and export images to the `images/` directory.

In [ ]:
# Define denormalization function
def denorm(x):
    y = (x + 1.0) / 2.0
    y = tf.clip_by_value(y, 0.0, 1.0)
    return y

# Define save JPEG function
def save_jpeg(tensor, path):
    img = tf.image.convert_image_dtype(tensor, dtype=tf.uint8)
    data = tf.io.encode_jpeg(img)
    tf.io.write_file(path, data)

# Define generator for all photos
def generate_all_photos(photo_dataset, max_images=None):
    count = 0
    for batch in photo_dataset.batch(8):
        generated = G(batch, training=False)
        generated = denorm(generated)
        for i in range(generated.shape[0]):
            if max_images is not None and count >= max_images:
                return count
            filename = os.path.join(OUTPUT_DIR, f'image_{count:05d}.jpg')
            save_jpeg(generated[i], filename)
            count += 1
    return count

total = generate_all_photos(photo_ds, max_images=None)
print('Generated images:', total)

# **[EDA of Generated Results]**
Visualize several generated painter-style images and compare them with the original photos.

In [ ]:
def show_generated_samples(photo_dataset, generator, num_samples=5):
    photos = next(iter(photo_dataset.batch(num_samples)))
    generated = generator(photos, training=False)
    generated = denorm(generated)

    plt.figure(figsize=(12, 4 * num_samples))
    for i in range(num_samples):
        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.imshow(((photos[i].numpy() + 1.0) / 2.0).clip(0, 1))
        plt.title("Original Photo")
        plt.axis("off")

        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.imshow(generated[i].numpy())
        plt.title("Generated Painter Style")
        plt.axis("off")

    plt.suptitle("Generated Painter-Style Results", fontsize=16)
    plt.tight_layout()
    plt.show()

show_generated_samples(photo_ds, G, num_samples=5)

# **[Save results]**

In [ ]:
import os
import glob
import zipfile

DRIVE_FOLDER = '/content/drive/MyDrive/CI/data'

os.makedirs(DRIVE_FOLDER, exist_ok=True)

zip_name = os.path.join(DRIVE_FOLDER, f'{PAINTER}_1_output_images.zip')
print(f"Zipping images to: {zip_name}")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.jpg'))):
        z.write(path, arcname=os.path.basename(path))

print('Success! Wrote zip to Google Drive:', zip_name)

# **[Results]**

In [ ]:
import os
import zipfile

ZIP_FILE_PATH = '/content/drive/MyDrive/CI/data/vangogh_1_output_images.zip'
OUTPUT_IMAGE_DIR = '/content/generated_images'

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

print(f"Unpacking images from {ZIP_FILE_PATH} to {OUTPUT_IMAGE_DIR}...")

if not os.path.exists(ZIP_FILE_PATH):
    print(f"Error: The zip file was not found at {ZIP_FILE_PATH}. Please ensure the file exists in your Google Drive and that Google Drive is correctly mounted.")
else:
    try:
        with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
            zip_ref.extractall(OUTPUT_IMAGE_DIR)
        print("Image extraction complete!")
    except Exception as e:
        print(f"An error occurred during extraction: {e}")

In [ ]:
results_ds = tf.keras.utils.image_dataset_from_directory(
    OUTPUT_IMAGE_DIR,
    labels=None,
    label_mode=None,
    color_mode='rgb',
    batch_size=1,
    shuffle=False,
    image_size=(256, 256)
)

results_ds = results_ds.map(lambda x: x / 255.0)
results_ds = results_ds.unbatch()

print("Results dataset created successfully.")

In [ ]:
import matplotlib.pyplot as plt
import math
import tensorflow as tf

def show_generated_samples(result_dataset, start_index=0, num_samples=20):
    """
    Plots generated result images starting from a specific index.

    Args:
        result_dataset: The TF dataset containing the images.
        start_index: The global index to start viewing from.
        num_samples: How many images to display.
    """
    cols = 3
    rows = math.ceil(num_samples / cols)

    try:
        subset = result_dataset.skip(start_index).batch(num_samples)
        results_batch = next(iter(subset))
    except StopIteration:
        print(f"Error: Not enough images found starting at index {start_index}.")
        return

    print(f"Displaying images {start_index} to {start_index + len(results_batch) - 1}")
    print(f"Batch shape: {results_batch.shape}")

    plt.figure(figsize=(15, 5 * rows))

    for i in range(num_samples):
        if i >= len(results_batch):
            break

        plt.subplot(rows, cols, i + 1)

        img = results_batch[i].numpy()

        if img.max() > 1.0:
            plt.imshow(img.astype("uint8"))
        else:
            plt.imshow(img.clip(0, 1))

        plt.title(f"Result Image {start_index + i}")
        plt.axis("off")

    plt.suptitle(f"Generated Results (Starting at {start_index})", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
show_generated_samples(results_ds, start_index=28, num_samples=21)